# Git Basics for AI Engineers

## Overview
Git is the universal version control system for code. For AI engineers it's foundational every experiment, every model version, every pipeline change should be tracked. This notebook covers git from first principles to daily workflows.

---

## 1. Version Control Concepts

**Why version control?**
- Track every change to your code over time
- Collaborate without overwriting each other's work
- Roll back to any previous state
- Understand *why* a change was made (commit messages)

**Git's 3-tree architecture:**
```
Working Directory  →  Staging Area (Index)  →  Repository (.git/)
   (modified)           (git add)              (git commit)
```

---

## 2. Core Git Commands

### Setup
```bash
git config --global user.name "Your Name"
git config --global user.email "you@example.com"
git config --global core.editor "code --wait"  # VS Code
```

### Repository Operations
```bash
git init                      # Initialize new repo
git clone <url>               # Clone remote repo
git clone <url> --depth 1     # Shallow clone (faster)
```

### Daily Workflow
```bash
git status                    # Check working tree status
git add file.py               # Stage specific file
git add .                     # Stage all changes
git add -p                    # Stage interactively (hunk by hunk)
git commit -m "feat: add RAG pipeline"   # Commit staged changes
git commit --amend            # Modify last commit (before push!)
```

### History & Diff
```bash
git log                       # Full history
git log --oneline --graph     # Compact graph view
git log --author="Alice"      # Filter by author
git diff                      # Unstaged changes
git diff --staged             # Staged changes
git diff main..feature-branch # Compare branches
git show abc1234              # Show specific commit
git blame file.py             # Who changed each line
```

### Branching
```bash
git branch                    # List local branches
git branch -a                 # List all branches (including remote)
git branch feature/rag        # Create branch
git checkout feature/rag      # Switch to branch
git checkout -b feature/rag   # Create AND switch
git switch -c feature/rag     # Modern equivalent
git branch -d feature/rag     # Delete merged branch
git branch -D feature/rag     # Force delete
```

### Merging & Rebasing
```bash
git merge feature/rag         # Merge into current branch
git merge --no-ff feature/rag # Always create merge commit
git rebase main               # Rebase current branch onto main
git rebase -i HEAD~3          # Interactive rebase (last 3 commits)
```

### Remote Operations
```bash
git remote -v                 # List remotes
git fetch origin              # Fetch without merging
git pull origin main          # Fetch + merge
git pull --rebase origin main # Fetch + rebase
git push origin feature/rag   # Push branch
git push -u origin feature/rag # Set upstream tracking
```

### Stashing
```bash
git stash                     # Stash dirty working directory
git stash push -m "WIP: RAG"  # Named stash
git stash list                # List all stashes
git stash pop                 # Apply and drop latest stash
git stash apply stash@{2}     # Apply specific stash
```

### Tags
```bash
git tag v1.0.0                # Lightweight tag
git tag -a v1.0.0 -m "Release 1.0" # Annotated tag
git push origin --tags        # Push all tags
```

### Undoing
```bash
git restore file.py           # Discard working directory changes
git restore --staged file.py  # Unstage
git revert abc1234            # Create reverse commit (safe)
git reset HEAD~1              # Undo last commit (keep changes)
git reset --hard HEAD~1       # Undo last commit (discard changes!)
git reflog                    # Safety net see all HEAD movements
```

---

## 3. Branching Strategies

### GitFlow
```
main ──────────────────────────────────────────→
         ↑ merge            ↑ merge
release/1.0──────────────────→
develop ──────────────────────────────────────→
         ↑ merge     ↑ merge
feature/A────→  feature/B────→
```
Good for: versioned software, complex release cycles.

### GitHub Flow (Simpler)
```
main ────────────────────────────────────→
       ↑ PR merge     ↑ PR merge
feature/A──→  feature/B──→
```
Good for: web apps, continuous deployment.

### Trunk-Based Development
Everyone commits to `main` frequently. Short-lived branches (< 1 day). Feature flags for incomplete work. Best for: CI/CD, high-velocity teams.

---

## 4. Merge vs Rebase

**Merge**: preserves full history with merge commits
```
A──B──C──M   ← merge commit
    ↘──D──↗
```

**Rebase**: linear history, rewrites commits
```
A──B──C──D'  ← D replayed on top of C
```

**Golden rule of rebase**: Never rebase public/shared branches.

---

## 5. .gitignore for ML Projects

```gitignore
# Python
__pycache__/
*.pyc
.venv/
*.egg-info/

# Jupyter
.ipynb_checkpoints/

# Data (large files)
data/raw/
data/processed/
*.csv
*.parquet

# Models (use DVC or Git LFS instead)
models/
*.pkl
*.pt
*.onnx

# Secrets
.env
*.pem
credentials.json

# MLflow
mlruns/

# W&B
wandb/
```

---

## 6. Conventional Commits

Structured commit messages for machine-readable changelogs:
```
type(scope): description

feat(rag): add hybrid search with BM25 + dense retrieval
fix(model): correct label encoder for unseen categories
docs(readme): update installation instructions
test(api): add integration tests for prediction endpoint
refactor(pipeline): extract feature engineering into separate module
perf(inference): batch tokenization for 3x speedup
chore(deps): upgrade transformers to 4.40.0
```

Types: `feat`, `fix`, `docs`, `style`, `refactor`, `test`, `chore`, `perf`, `ci`

In [1]:
# ── Git commands via Python (for scripting) ─────────────────────
import subprocess

def git(cmd: str, cwd=None) -> str:
    """Run a git command and return output."""
    result = subprocess.run(
        ['git'] + cmd.split(),
        capture_output=True, text=True, cwd=cwd
    )
    if result.returncode != 0:
        return f"Error: {result.stderr.strip()}"
    return result.stdout.strip()

# Get current git info
import os
repo_path = '/home/dell/Desktop'

# Check git version
version = git('--version')
print(f"Git version: {version}")

# Show git config
name = git('config user.name')
email = git('config user.email')
print(f"User: {name} <{email}>")

Git version: git version 2.43.0
User: Error:  <ibtasam@visnext.net>


In [2]:
# ── pre-commit hooks setup ──────────────────────────────────────
# pip install pre-commit

pre_commit_config = """
# .pre-commit-config.yaml
repos:
  - repo: https://github.com/pre-commit/pre-commit-hooks
    rev: v4.5.0
    hooks:
      - id: trailing-whitespace
      - id: end-of-file-fixer
      - id: check-yaml
      - id: check-json
      - id: check-added-large-files
        args: ['--maxkb=500']
      - id: detect-private-key

  - repo: https://github.com/psf/black
    rev: 24.3.0
    hooks:
      - id: black

  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.3.5
    hooks:
      - id: ruff
        args: [--fix]

  - repo: https://github.com/pre-commit/mirrors-mypy
    rev: v1.9.0
    hooks:
      - id: mypy
"""

print("Save as .pre-commit-config.yaml, then run:")
print("  pre-commit install   # Install hooks")
print("  pre-commit run --all-files  # Run on all files")
print()
print(pre_commit_config)

Save as .pre-commit-config.yaml, then run:
  pre-commit install   # Install hooks
  pre-commit run --all-files  # Run on all files


# .pre-commit-config.yaml
repos:
  - repo: https://github.com/pre-commit/pre-commit-hooks
    rev: v4.5.0
    hooks:
      - id: trailing-whitespace
      - id: end-of-file-fixer
      - id: check-yaml
      - id: check-json
      - id: check-added-large-files
        args: ['--maxkb=500']
      - id: detect-private-key

  - repo: https://github.com/psf/black
    rev: 24.3.0
    hooks:
      - id: black

  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.3.5
    hooks:
      - id: ruff
        args: [--fix]

  - repo: https://github.com/pre-commit/mirrors-mypy
    rev: v1.9.0
    hooks:
      - id: mypy



In [3]:
# ── Resolving merge conflicts ────────────────────────────────────
conflict_example = """
When a merge conflict occurs, the file looks like:

<<<<<<< HEAD (your branch)
model = RandomForestClassifier(n_estimators=200)
=======
model = XGBClassifier(n_estimators=100, learning_rate=0.1)
>>>>>>> feature/xgboost

Resolution steps:
1. Open the file and choose which version to keep (or combine both)
2. Remove the conflict markers (<<<<<<<, =======, >>>>>>>)
3. git add <resolved-file>
4. git commit

Useful tools:
  git mergetool           # Opens visual merge tool
  git checkout --ours .   # Take all our changes
  git checkout --theirs . # Take all their changes
"""
print(conflict_example)


When a merge conflict occurs, the file looks like:

<<<<<<< HEAD (your branch)
model = RandomForestClassifier(n_estimators=200)
model = XGBClassifier(n_estimators=100, learning_rate=0.1)
>>>>>>> feature/xgboost

Resolution steps:
1. Open the file and choose which version to keep (or combine both)
2. Remove the conflict markers (<<<<<<<, =======, >>>>>>>)
3. git add <resolved-file>
4. git commit

Useful tools:
  git mergetool           # Opens visual merge tool
  git checkout --ours .   # Take all our changes
  git checkout --theirs . # Take all their changes



## Additional Learning Resources

### Books & References
- [Pro Git Book (free)](https://git-scm.com/book/en/v2) Comprehensive reference
- [Oh Shit, Git!?!](https://ohshitgit.com/) Fix common git mistakes
- [Git Cheat Sheet (GitHub)](https://education.github.com/git-cheat-sheet-education.pdf)

### Interactive Learning
- [Learn Git Branching (visual)](https://learngitbranching.js.org/)
- [Git Immersion](https://gitimmersion.com/)
- [Atlassian Git Tutorials](https://www.atlassian.com/git/tutorials)

### Workflow References
- [GitHub Flow](https://docs.github.com/en/get-started/quickstart/github-flow)
- [Conventional Commits Spec](https://www.conventionalcommits.org/)
- [pre-commit docs](https://pre-commit.com/)